# Typed Failure Detector & Type-1/Type-3 Repairs for Text-to-FOL Neuro-Symbolic Pipelines

## Overview

This notebook demonstrates a **neuro-symbolic pipeline** that converts natural language text into first-order logic (Prolog), executes logical queries, classifies proof failures into **typed categories**, and dispatches **type-specific LLM repairs**.

### Key Components

1. **Prolog Backward-Chaining Engine**: Custom Python implementation with unification and term parsing
2. **Failure Classification**: Maps proof failures to 5 typed categories (lexical mismatch, arity mismatch, missing fact, type violation, scope conflict)
3. **Type-Specific Repairs**:
   - **TYPE_1**: Generate bridge axioms linking mismatched predicates
   - **TYPE_3**: Abductive fact injection with source-span verification
4. **Bridge Axiom Library**: Accumulates reusable axioms across documents for improved reuse
5. **Baseline**: ARGOS single-strategy abductive repair (uniform approach regardless of failure type)

### Evaluation

- **Datasets**: RuleTaker (propositional logic chains) and CLUTRR (kinship multi-hop reasoning)
- **Metrics**: Query accuracy, hallucination rates, failure type distribution
- **Output**: Structured results with predictions, failure types, and repair traces


In [ ]:
# Install and import dependencies
import subprocess, sys
print(f"Python {sys.version}")

# Try importing core packages; they're pre-installed on Colab
try:
    import pandas as pd
    import matplotlib.pyplot as plt
    print("✓ pandas, matplotlib already available")
except ImportError:
    print("Note: pandas/matplotlib not pre-installed; visualization cells will be skipped")
    pd = None
    plt = None

print("Core setup complete.")

In [ ]:
# Core imports
import json
import re
from dataclasses import dataclass
from pathlib import Path
from typing import Optional, Union, Generator, Any
from collections import defaultdict

print("Core imports loaded.")

In [ ]:
# Data loading helper
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-1f4229-typed-unification-failure-recovery-towar/main/round-1/experiment-1/demo/mini_demo_data.json"

def load_data():
    """Load demo data from GitHub URL with local fallback."""
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception as e:
        pass  # Try local file
    
    local_path = Path("mini_demo_data.json")
    if local_path.exists():
        with open(local_path) as f:
            return json.load(f)
    
    raise FileNotFoundError("Could not load mini_demo_data.json from GitHub or local filesystem")

print("Data loading helper defined.")

In [ ]:
# Load data
data = load_data()
print(f"✓ Data loaded. Metadata: {data['metadata']['method_name']}")
print(f"✓ Datasets: {[d['dataset'] for d in data['datasets']]}")

## Configuration

Define tunable parameters for the demo. All are set to minimal values for fast iteration. Increase them to see more detailed results.

In [ ]:
# Demo configuration: minimal values for fast execution
NUM_EXAMPLES_TO_DISPLAY = 2  # Show first N results per dataset
SHOW_FULL_EXAMPLES = False   # If True, show complete input/output; if False, truncate
MAX_INPUT_LENGTH = 150       # Truncate long inputs for readability

print(f"✓ Config: Display {NUM_EXAMPLES_TO_DISPLAY} examples, truncate inputs to {MAX_INPUT_LENGTH} chars")

## Prolog Engine: Core Data Structures

This section implements the **Prolog backward-chaining engine** with unification. The engine parses logical clauses, indexes them by predicate/arity, and performs proof search with variable renaming.


In [ ]:
# Prolog term types: Variables, Atoms, and Compound terms
@dataclass(frozen=True)
class Var:
    """Prolog variable (uppercase or _)."""
    name: str
    def __repr__(self) -> str:
        return self.name

@dataclass(frozen=True)
class Atom:
    """Prolog atom (lowercase constant or quoted string)."""
    name: str
    def __repr__(self) -> str:
        return self.name

@dataclass(frozen=True)
class Compound:
    """Prolog compound term: functor(arg1, arg2, ...)."""
    functor: str
    args: tuple
    @property
    def arity(self) -> int:
        return len(self.args)
    def __repr__(self) -> str:
        return f"{self.functor}({', '.join(repr(a) for a in self.args)})"

Term = Union[Var, Atom, Compound]
print("✓ Prolog term types defined.")

In [ ]:
# Core unification algorithm: Robinson's algorithm with occurs check omitted for speed

def walk(t: Term, env: dict) -> Term:
    """Follow variable bindings in the environment."""
    while isinstance(t, Var) and t in env:
        t = env[t]
    return t

def unify(t1: Term, t2: Term, env: dict) -> Optional[dict]:
    """Unify two terms, returning updated bindings or None if unification fails."""
    t1 = walk(t1, env)
    t2 = walk(t2, env)
    if t1 == t2:
        return env
    if isinstance(t1, Var):
        return {**env, t1: t2}
    if isinstance(t2, Var):
        return {**env, t2: t1}
    if isinstance(t1, Compound) and isinstance(t2, Compound):
        if t1.functor != t2.functor or t1.arity != t2.arity:
            return None
        new_env = env
        for a1, a2 in zip(t1.args, t2.args):
            new_env = unify(a1, a2, new_env)
            if new_env is None:
                return None
        return new_env
    return None

def apply_env(t: Term, env: dict) -> Term:
    """Apply bindings from the environment to a term."""
    t = walk(t, env)
    if isinstance(t, (Var, Atom)):
        return t
    if isinstance(t, Compound):
        return Compound(t.functor, tuple(apply_env(a, env) for a in t.args))
    return t

print("✓ Unification algorithm implemented.")

In [ ]:
# Parsing: convert Prolog syntax to term structures

def split_args(s: str) -> list[str]:
    """Split comma-separated arguments, respecting nested parentheses."""
    depth, current, args = 0, "", []
    for c in s:
        if c == '(':
            depth += 1
            current += c
        elif c == ')':
            depth -= 1
            current += c
        elif c == ',' and depth == 0:
            args.append(current.strip())
            current = ""
        else:
            current += c
    if current.strip():
        args.append(current.strip())
    return args

def parse_term(s: str) -> Term:
    """Parse a Prolog term: atom, variable, or compound."""
    s = s.strip()
    if not s:
        return Atom("nil")
    if s.startswith("'") and s.endswith("'"):
        return Atom(s[1:-1])
    if s[0].isupper() or s[0] == '_':
        return Var(s)
    m = re.match(r'^([a-zA-Z_][a-zA-Z0-9_]*)\((.+)\)$', s, re.DOTALL)
    if m:
        return Compound(m.group(1), tuple(parse_term(a) for a in split_args(m.group(2))))
    return Atom(s.replace(' ', '_').replace('-', '_').lower())

def parse_clause(s: str) -> Optional[tuple[Term, list[Term]]]:
    """Parse a Prolog clause: head or head :- body."""
    s = re.sub(r'%[^\n]*', '', s).strip().rstrip('.')
    if not s:
        return None
    if ':-' in s:
        head_str, body_str = s.split(':-', 1)
        head = parse_term(head_str.strip())
        body = [parse_term(g.strip()) for g in split_args(body_str.strip())]
    else:
        head = parse_term(s)
        body = []
    return head, body

print("✓ Prolog parser implemented.")

In [ ]:
# Knowledge base: stores clauses and provides indexed query interface

class PrologKB:
    """Prolog knowledge base with backward-chaining proof search."""
    
    def __init__(self):
        self.clauses: list[tuple[Term, list[Term]]] = []
        self.index: dict[tuple[str, int], list[int]] = {}  # (functor, arity) -> clause indices

    def _key(self, head: Term) -> tuple[str, int]:
        """Extract (functor, arity) key from a head term."""
        if isinstance(head, Compound):
            return (head.functor, head.arity)
        if isinstance(head, Atom):
            return (head.name, 0)
        return ("__unknown__", 0)

    def add_clause(self, head: Term, body: list[Term]):
        """Add a clause and index it by predicate/arity."""
        idx = len(self.clauses)
        self.clauses.append((head, body))
        key = self._key(head)
        self.index.setdefault(key, []).append(idx)

    def load_str(self, s: str) -> bool:
        """Parse and add a single clause."""
        try:
            result = parse_clause(s)
            if result:
                self.add_clause(*result)
                return True
        except Exception:
            pass
        return False

    def predicates(self) -> set[tuple[str, int]]:
        """Return set of (functor, arity) pairs in KB."""
        return set(self.index.keys())

    def query(self, goal_str: str, max_depth: int = 40) -> tuple[bool, dict, Optional[str]]:
        """Query the KB. Returns (success, bindings, exception_signal)."""
        try:
            goal = parse_term(goal_str.strip().rstrip('.'))
        except Exception as e:
            return False, {}, f"parse_error({e})"

        if isinstance(goal, Compound):
            key = (goal.functor, goal.arity)
        elif isinstance(goal, Atom):
            key = (goal.name, 0)
        else:
            return False, {}, "type_error(callable)"

        if key not in self.index:
            same = [k for k in self.index if k[0] == key[0]]
            if same:
                return False, {}, f"existence_error(arity_mismatch,{key[0]}/{key[1]},expected/{same[0][1]})"
            return False, {}, f"existence_error(procedure,{key[0]}/{key[1]})"

        try:
            for bindings in self._prove([goal], {}, 0, max_depth):
                return True, {str(k): str(apply_env(k, bindings)) for k in bindings}, None
        except RecursionError:
            return False, {}, "resource_error(max_depth)"

        return False, {}, None  # Proof exhausted (TYPE_3)

    def _prove(self, goals: list[Term], env: dict, depth: int, max_depth: int):
        """Recursive backward-chaining proof search."""
        if depth > max_depth:
            return
        if not goals:
            yield env
            return
        goal = apply_env(goals[0], env)
        rest = goals[1:]
        key = (goal.functor, goal.arity) if isinstance(goal, Compound) else \
              (goal.name, 0) if isinstance(goal, Atom) else None
        if not key or key not in self.index:
            return
        for idx in self.index[key]:
            head, body = self.clauses[idx]
            sfx = f"${depth}_{idx}"
            def ren(t: Term) -> Term:
                if isinstance(t, Var):
                    return Var(t.name + sfx)
                if isinstance(t, Compound):
                    return Compound(t.functor, tuple(ren(a) for a in t.args))
                return t
            new_env = unify(goal, ren(head), env)
            if new_env is not None:
                yield from self._prove([ren(g) for g in body] + list(rest), new_env, depth + 1, max_depth)

print("✓ PrologKB class implemented.")

## Quick Prolog Test

Test the Prolog engine on a simple example from RuleTaker.


In [ ]:
# Test example from RuleTaker
kb = PrologKB()
clauses = [
    "likes(lion, cow).",
    "likes(cat, bird).",
    "is_cold(cow).",
    "is_cold(X) :- likes(X, Y), is_cold(Y)."
]

for clause in clauses:
    kb.load_str(clause)

print(f"✓ Loaded {len(kb.clauses)} clauses")
print(f"✓ Predicates: {kb.predicates()}")

# Query: Is Lion cold?
success, bindings, signal = kb.query("is_cold(lion)")
print(f"\nQuery: is_cold(lion)")
print(f"  Success: {success}")
print(f"  Signal: {signal}")
print(f"  Bindings: {bindings}")

## Failure Classification

When a Prolog proof fails, we classify the failure into **5 typed categories**:

1. **TYPE_1_LEXICAL_MISMATCH**: Predicate absent from KB, but a similar predicate exists
2. **TYPE_2_ARITY_MISMATCH**: Predicate exists with different arity
3. **TYPE_3_MISSING_FACT**: Proof search exhausted, need an additional fact
4. **TYPE_4_CATEGORY_VIOLATION**: Type error (non-callable goal)
5. **TYPE_5_SCOPE_CONFLICT**: Heuristic: deeply nested goal indicating quantifier scope issue

Each type suggests a different repair strategy.


In [ ]:
def _jaccard(a: str, b: str) -> float:
    """Jaccard similarity between two strings (character-level)."""
    sa, sb = set(a.lower()), set(b.lower())
    u = len(sa | sb)
    return len(sa & sb) / u if u else 0.0

def classify_failure(signal: Optional[str], goal: str, predicates: set[tuple[str, int]]) -> dict:
    """Classify a proof failure into a typed category."""
    if signal is None:
        return {"type": "TYPE_3_MISSING_FACT", "goal": goal}

    if "arity_mismatch" in signal:
        m = re.search(r'(\w+)/(\d+).*expected/(\d+)', signal)
        if m:
            return {
                "type": "TYPE_2_ARITY_MISMATCH", "goal": goal,
                "predicate": m.group(1), "called": int(m.group(2)), "expected": int(m.group(3)),
            }

    if "existence_error(procedure" in signal:
        m = re.search(r'procedure,(\w+)/(\d+)', signal)
        missing = m.group(1) if m else "unknown"
        candidates = [
            f"{f}/{a}" for f, a in predicates
            if f != missing and _jaccard(f, missing) > 0.5
        ]
        return {
            "type": "TYPE_1_LEXICAL_MISMATCH", "goal": goal,
            "missing_pred": missing, "candidates": candidates[:3],
        }

    if "type_error" in signal:
        return {"type": "TYPE_4_CATEGORY_VIOLATION", "goal": goal, "signal": signal}

    if goal.count('(') > 3:
        return {"type": "TYPE_5_SCOPE_CONFLICT", "goal": goal}

    return {"type": "UNKNOWN", "goal": goal, "signal": signal}

print("✓ Failure classifier implemented.")

In [ ]:
# Test failure classification
kb2 = PrologKB()
kb2.load_str("likes(lion, cow).")

success, bindings, signal = kb2.query("unknown_pred(lion)")
failure = classify_failure(signal, "unknown_pred(lion)", kb2.predicates())

print(f"Query: unknown_pred(lion)")
print(f"  Success: {success}")
print(f"  Signal: {signal}")
print(f"  \nFailure Classification:")
print(f"    Type: {failure['type']}")
if 'missing_pred' in failure:
    print(f"    Missing: {failure['missing_pred']}")
    print(f"    Candidates: {failure.get('candidates', [])}")

## Results Analysis

Load the demo data and analyze the experimental results. Show:
- Metrics (accuracy, hallucination rates) per dataset and failure type
- Sample examples with predictions and metadata


In [ ]:
# Extract and display metadata
metadata = data['metadata']
print(f"\n{'='*60}")
print(f"EXPERIMENT METADATA")
print(f"{'='*60}")
print(f"Method: {metadata['method_name']}")
print(f"LLM Model: {metadata['llm_model']}")
print(f"Total LLM Cost: ${metadata['total_cost_usd']}")
print(f"Bridge Axiom Library: {metadata['bridge_axiom_library_size']} axioms")
print(f"Reuse Rate: {metadata['bridge_axiom_reuse_rate']:.2f}")
print()

# Display metrics per dataset
for dataset_name, metrics in metadata['metrics'].items():
    print(f"\n{'-'*60}")
    print(f"{dataset_name.upper()} DATASET (n={metrics['n']})")
    print(f"{'-'*60}")
    print(f"Accuracy:")
    print(f"  Typed:     {metrics['typed_accuracy']:.1%}")
    print(f"  Baseline:  {metrics['baseline_accuracy']:.1%}")
    print(f"  Δ:         {metrics['improvement_pct']:+.1f}pp")
    print(f"\nHallucination Rate:")
    print(f"  Typed:     {metrics['typed_hallucination_rate']:.1%}")
    print(f"  Baseline:  {metrics['baseline_hallucination_rate']:.1%}")
    print(f"  Reduction: {metrics['hallucination_reduction_pct']:.1f}%")
    print(f"\nFailure Type Distribution:")
    for ftype, count in sorted(metrics['failure_distribution'].items()):
        pct = count / metrics['n'] * 100
        print(f"  {ftype:30s} {count:3d} ({pct:5.1f}%)")

In [ ]:
# Display sample examples per dataset
for dataset_group in data['datasets']:
    dataset_name = dataset_group['dataset']
    examples = dataset_group['examples'][:NUM_EXAMPLES_TO_DISPLAY]
    
    print(f"\n{'='*60}")
    print(f"{dataset_name.upper()} EXAMPLES")
    print(f"{'='*60}")
    
    for i, ex in enumerate(examples, 1):
        print(f"\nExample {i}:")
        
        # Input
        inp = ex['input']
        if not SHOW_FULL_EXAMPLES and len(inp) > MAX_INPUT_LENGTH:
            inp = inp[:MAX_INPUT_LENGTH] + "..."
        print(f"  Input: {inp}")
        
        # Gold label
        print(f"  Gold: {ex['output']}")
        
        # Predictions
        print(f"  Predictions:")
        typed_ok = ex['metadata_typed_correct']
        baseline_ok = ex['metadata_baseline_correct']
        print(f"    Typed: {ex['predict_typed']} ({'✓' if typed_ok == 'True' else '✗'})")
        print(f"    Baseline: {ex['predict_baseline']} ({'✓' if baseline_ok == 'True' else '✗'})")
        
        # Metadata
        print(f"  Metadata:")
        print(f"    Goal: {ex['metadata_goal']}")
        print(f"    Failure Type: {ex['metadata_failure_type']}")
    print()

## Summary

This demo notebook demonstrates a **typed failure detection and repair system** for neuro-symbolic reasoning:

### Key Findings

1. **RuleTaker (Propositional Logic)**: Both typed and baseline achieve 95% accuracy. Typed repairs reduce hallucination by 35.5%, showing better fact precision even when both succeed.

2. **CLUTRR (Kinship Relations)**: 0% accuracy due to lack of relational variable binding in the goal extractor—a known limitation requiring variable-scoped unification. Both methods fail equally, but typed repairs still reduce hallucination by 32.3%.

3. **Typed Failure Classification**: Enables targeted repairs:
   - **TYPE_1**: Bridge axioms for predicate mismatches
   - **TYPE_3**: Abductive fact injection for missing facts
   - Shared bridge library accumulates reusable axioms across documents

4. **Hallucination Reduction**: Even when accuracy is identical, typed repairs with source-span verification reduce hallucinated clauses compared to baseline single-strategy repairs.

### Technical Architecture

- **Unification**: Robinson's algorithm with environment-based variable binding
- **Proof Search**: Backward-chaining with depth-bounded search and clause indexing by predicate/arity
- **Failure Signals**: Exception codes from KB query (parse errors, arity mismatches, missing predicates)
- **Repairs**: Type-specific LLM prompts that cite source sentences for grounding

### Future Work

- Implement relational variable binding for CLUTRR
- Expand bridge axiom reuse across larger document corpora
- Integrate with knowledge graphs for semantic grounding
